In [1]:
import torch
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/mnt/f/whisper-finetune/env_whisper-finetune/lib/python3.9/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/mnt/f/whisper-finetune/env_whisper-finetune/lib/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/mnt/f/whisper-finetune/env_whisper-finetune/lib/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/mnt/f/whisper-finetune/env_whisper-finetune/lib/python3.9/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should encode the Optional
  File "/mnt/f/whisper-finetune/env_whisper-finetune/lib/python3.9/site-packages/t

/mnt/f/whisper-finetune/env_whisper-finetune/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset = load_dataset("changelinglab/easycall-dysarthria")

print(dataset)
print(dataset["train"][0])

Found cached dataset parquet (/home/hillol/.cache/huggingface/datasets/changelinglab___parquet/changelinglab--easycall-dysarthria-40615a80a37ddc0a/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)
100%|██████████| 3/3 [00:00<00:00, 39.21it/s]


DatasetDict({
    test: Dataset({
        features: ['audio', 'filename', 'speaker', 'text', 'dysarthria_severity'],
        num_rows: 5213
    })
    validation: Dataset({
        features: ['audio', 'filename', 'speaker', 'text', 'dysarthria_severity'],
        num_rows: 4272
    })
    train: Dataset({
        features: ['audio', 'filename', 'speaker', 'text', 'dysarthria_severity'],
        num_rows: 11901
    })
})


In [3]:
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [4]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="italian",
    task="transcribe"
)

In [5]:
from datasets import disable_caching
disable_caching()

In [6]:
# def prepare(batch):
#     audio = batch["audio"]

#     # input features (log-Mel spectrogram)
#     batch["input_features"] = processor.feature_extractor(
#         audio["array"],
#         sampling_rate=audio["sampling_rate"]
#     ).input_features[0]

#     # labels (tokenized text)
#     batch["labels"] = processor.tokenizer(
#         batch["text"],
#         truncation=True
#     ).input_ids

#     return batch
def prepare(batch):
    audio_arrays = [x["array"] for x in batch["audio"]]
    sampling_rates = [x["sampling_rate"] for x in batch["audio"]]

    # input features
    batch["input_features"] = processor.feature_extractor(
        audio_arrays,
        sampling_rate=sampling_rates[0],  # all same after cast
    ).input_features

    # labels
    batch["labels"] = processor.tokenizer(
        batch["text"],
        padding=True,
        truncation=True
    ).input_ids

    return batch

In [7]:
# dataset = dataset.map(
#     prepare,
#     remove_columns=dataset["train"].column_names,
#     num_proc=2
# )
dataset = dataset.with_transform(prepare)

In [8]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="italian",
    task="transcribe"
)

In [9]:
for param in model.model.encoder.parameters():
    param.requires_grad = False

In [10]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-dysarthria-it",

    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=5e-6,
    warmup_steps=100,

    num_train_epochs=10,

    fp16=True,

    evaluation_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,

    predict_with_generate=True,
    generation_max_length=128,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    dataloader_num_workers=0,
    remove_unused_columns=False,
    
)

In [12]:
import setuptools
import distutils.version

In [15]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):

        # separate inputs and labels
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        # pad inputs
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        # pad labels
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        # replace padding with -100 for loss masking
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id,
            -100
        )

        batch["labels"] = labels

        return batch

In [16]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [17]:
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Wer
1,0.076300,0.068242,25.130729
2,0.048700,0.052903,31.161082
3,0.034000,0.053464,32.615177


TrainOutput(global_step=2232, training_loss=0.3756671960609147, metrics={'train_runtime': 8849.3419, 'train_samples_per_second': 13.448, 'train_steps_per_second': 0.841, 'total_flos': 1.030336454762496e+19, 'train_loss': 0.3756671960609147, 'epoch': 3.0})

In [19]:
trainer.evaluate(dataset["test"])

{'eval_loss': 0.10408914089202881,
 'eval_wer': 26.06307409675444,
 'eval_runtime': 2095.9081,
 'eval_samples_per_second': 2.487,
 'eval_steps_per_second': 0.311,
 'epoch': 3.0}

In [ ]:
sample = dataset["test"][0]

input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to("cuda")

pred_ids = model.generate(input_features)

print("PRED:", processor.batch_decode(pred_ids, skip_special_tokens=True)[0])

KeyboardInterrupt: 